In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
h_big = 5
w_big = 5

h_small = 3
w_small = 3

avg_len = 0.2

In [ ]:
np.linspace(0.03, 0.2, 18)

In [ ]:
ipu, points, segment_edges, m, marker= periodic_unit_helper.get_boundary_aligned_dashline(w_small, w_big, h_small, h_big, avg_len)

In [ ]:
visualization.plot_line_segments(points, segment_edges)

In [ ]:
finalMarkers = np.where(np.array(marker) == 1)[0]

In [ ]:
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# # m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)


# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)
# # m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)


In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
len(m.vertices())

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
fuse_boundary = False

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
ipu.get_alpha()

In [ ]:
viewer.update()



In [ ]:
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])



In [ ]:
np.mean(ipu.getVars()[3:-2].reshape((int((ipu.numVars() - 5) / 3), 3))[:, 2])



In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [ipu.numVars() - 2], 1e-6
# fixedVars, hessianShift = [], 1e-6

In [ ]:
disableFusedRegionTFT = False

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 3

In [ ]:
# ipu.gradient()

In [ ]:
ipu.energy()

In [ ]:
benchmark.reset()

opts.niter = 1000
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
optimizer = inflation.get_inflation_optimizer(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr = optimizer.optimize()
benchmark.report()

In [ ]:
cr.success

In [ ]:
ipu.get_alpha()

In [ ]:
ipu.get_kappa()

In [ ]:
import importlib

In [ ]:
import periodic_simulation_setup
importlib.reload(periodic_simulation_setup)
importlib.reload(periodic_unit_helper)
import periodic_simulation_setup

In [ ]:

periodic_simulation_setup.reparametrize_gamma_bar(ipu)

In [ ]:
ipu.energy()

### Convert to zero average z offset structure

In [ ]:
    reparametrize_gamma_bar(ipu)


In [ ]:
ipu.energy()

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, disableFusedRegionTFT)

In [ ]:
az_ipu.energy()

In [ ]:
fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
optimizer

In [ ]:
benchmark.reset()
stiffness, sampled_alphas = periodic_simulation_setup.visualize_sampled_bending_stiffness(az_ipu, 100, az_optimizer, filename = "stiffness_shifted_dashline_{}_resolution_{}_disableTFT_{}.png".format(0, avg_len, disableFusedRegionTFT), hessianShift=1e-5)
benchmark.report()

In [ ]:
stiffness

In [ ]:
max(stiffness), min(stiffness)

In [ ]:
perturb = np.zeros(ipu.numVars())
perturb[-2] = 1e-6

In [ ]:
# perturb = np.zeros(ipu.numVars())
# perturb[-2] = 1e-6
# fd_validation.secondDerivativeConvergencePlot(periodic_simulation_setup.bending_stiffness_class(ipu, ipu.sheet, viewer), perturb = perturb, epsilons = np.logspace(1, 5, 100))